In [15]:
from dotenv import load_dotenv
load_dotenv("../.env")  # load the repo-root .env so this works outside VS Code too
from decouple import config

In [16]:
assert config("POLYGON_API_KEY", default=None, cast=str) is not None

In [17]:
import requests
POLYGON_API_KEY = config("POLYGON_API_KEY", default=None, cast=str)

In [18]:
ticker = "AAPL" #the particular stock
multiplier = "5" #the interval of the data
timespan = "minute" #the time frame
from_date = "2026-10-16" #from date
to_date = "2026-10-16" #end date
path = f"/v2/aggs/ticker/{ticker}/range/{multiplier}/{timespan}/{from_date}/{to_date}"
url = f"https://api.polygon.io{path}?apiKey={POLYGON_API_KEY}"

In [23]:
import pytz
from dataclasses import dataclass
from typing import Literal
from urllib.parse import urlencode
from datetime import datetime

POLYGON_API_KEY = config("POLYGON_API_KEY", default=None, cast=str)



def transform_polygon_result(result):
    unix_timestamp = result.get('t') / 1000.0
    utc_timestamp = datetime.fromtimestamp(unix_timestamp, tz=pytz.timezone('UTC'))
    return {
        'open_price': result['o'],
        'close_price': result['c'],
        'high_price': result['h'],
        'low_price': result['l'],
        'number_of_trades': result['n'],
        'volume': result['v'],
        'volume_weighted_average': result['vw'],
        'time': utc_timestamp,
    }

#dataclass
@dataclass
class PolygonAPIClient:
    ticker: str = "AAPL" #apple stock
    multiplier: int = 5 #interval of the data
    timespan:str = "minute" #time frame
    from_date:str = "2026-10-16" #from date
    to_date:str = "2026-10-16" #end date
    api_key:str = "" #api key
    adjusted: bool = True 
    sort: Literal["asc", "desc"] = "asc"

    def get_api_key(self):
        return self.api_key or POLYGON_API_KEY

    def get_headers(self):
        api_key = self.get_api_key()
        return {
            "Authorization": f"Bearer {api_key}"
        }

    def get_params(self):
        return {
            "adjusted": self.adjusted,
            "sort": self.sort
        }
    
    def generate_url(self, pass_auth=False):
        path = f"/v2/aggs/ticker/{self.ticker}/range/{self.multiplier}/{self.timespan}/{self.from_date}/{self.to_date}"
        url = f"https://api.polygon.io{path}"
        params = self.get_params()
        encoded_params = urlencode(params)
        url = f"{url}?{encoded_params}"
        if pass_auth:
            api_key = self.get_api_key()
            url += f"&api_key={api_key}"
        print(f"Generated URL: {url}")  # Debugging statement
        return url

    def fetch_data(self):
        headers = self.get_headers()
        url = self.generate_url()
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status() # not 200/201
        return response.json()

    def get_stock_data(self):
        data = self.fetch_data()
        results = data['results']
        dataset = []
        for result in results:
            dataset.append(
                transform_polygon_result(result)
            )
        return dataset
        

In [24]:
stock_api_client = PolygonAPIClient(ticker="GOOG", multiplier=1)
stock_api_client.generate_url(pass_auth=True)

Generated URL: https://api.polygon.io/v2/aggs/ticker/GOOG/range/1/minute/2026-10-16/2026-10-16?adjusted=True&sort=asc&api_key=c01BAKdBE6eIeMxb7Udl7mZmYGakDkRw


'https://api.polygon.io/v2/aggs/ticker/GOOG/range/1/minute/2026-10-16/2026-10-16?adjusted=True&sort=asc&api_key=c01BAKdBE6eIeMxb7Udl7mZmYGakDkRw'

In [21]:
dataset = stock_api_client.get_stock_data()
dataset

[{'open_price': 344.7465,
  'close_price': 344.91,
  'high_price': 345.0601,
  'low_price': 344.27,
  'number_of_trades': 239,
  'volume': 7590.135051,
  'volume_weighted_average': 344.7561,
  'time': datetime.datetime(2026, 10, 16, 8, 0, tzinfo=<UTC>)},
 {'open_price': 344.72,
  'close_price': 344.86,
  'high_price': 345.46,
  'low_price': 344.29,
  'number_of_trades': 497,
  'volume': 9212.015818,
  'volume_weighted_average': 344.7034,
  'time': datetime.datetime(2026, 10, 16, 8, 1, tzinfo=<UTC>)},
 {'open_price': 344.49,
  'close_price': 344.65,
  'high_price': 345.5,
  'low_price': 344.25,
  'number_of_trades': 539,
  'volume': 11089,
  'volume_weighted_average': 344.6702,
  'time': datetime.datetime(2026, 10, 16, 8, 2, tzinfo=<UTC>)},
 {'open_price': 345.8693,
  'close_price': 345.8221,
  'high_price': 345.8693,
  'low_price': 344.49,
  'number_of_trades': 13,
  'volume': 501.017382,
  'volume_weighted_average': 345.6629,
  'time': datetime.datetime(2026, 10, 16, 8, 4, tzinfo=<UTC